In [9]:

!pip -q install transformers torch


In [10]:

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

# 1. Select model and device
MODEL_NAME = "gpt2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 2. Create a custom text dataset
texts = [
    "Artificial intelligence helps computers perform intelligent tasks.",
    "Machine learning allows computers to learn from data.",
    "Natural language processing helps computers understand human language.",
    "Deep learning uses neural networks to solve complex problems.",
    "GPT-2 generates text by predicting the next token.",
    "Transformers use attention mechanisms to process language.",
    "Artificial intelligence is used in healthcare and education.",
    "Language models can generate stories, articles, and conversations.",
    "Python is a popular programming language for machine learning.",
    "Text generation is an important application of natural language processing."
] * 10

# 3. Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

# 4. Prepare custom dataset
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=64):
        self.examples = []

        for text in texts:
            encoded = tokenizer(
                text,
                max_length=max_length,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            self.examples.append({
                "input_ids": encoded["input_ids"].squeeze(0),
                "attention_mask": encoded["attention_mask"].squeeze(0)
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

dataset = TextDataset(texts, tokenizer)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# 5. Fine-tune GPT-2
optimizer = AdamW(model.parameters(), lr=5e-5)
epochs = 2

print("\nStarting training...")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {total_loss / len(loader):.4f}"
    )

# 6. Generate new text
model.eval()

prompt = "Artificial intelligence"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("\nInput Prompt:")
print(prompt)

print("\nGenerated Text:")
print(generated_text)

# 7. Save the fine-tuned model
model.save_pretrained("./custom_gpt2_model")
tokenizer.save_pretrained("./custom_gpt2_model")

print("\nModel saved successfully!")


Device: cpu


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Starting training...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1/2, Loss: 1.1871
Epoch 2/2, Loss: 0.1963

Input Prompt:
Artificial intelligence

Generated Text:
Artificial intelligence helps computers perform intelligent tasks.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!
